# Akıllı Ev Sistemleri için Saldırı Tespit Sistemi — Akademik Karşılaştırma

**Proje:** Akıllı Ev Sistemlerine Yönelik Saldırı Tespit Sistemi (Kısıtlı Donanım ve SLM Odağında)  
**Yazar:** Galip Talha ERBAŞ — Muğla Sıtkı Koçman Üniversitesi  
**Repo:** https://github.com/galiperbas/HIDS-XGBoost-SHAP-SLM

---

## ⚙️ Çalıştırma Ortamı — ZORUNLU Ayarlar

**Runtime → Change runtime type:**
- **Donanım hızlandırıcı:** L4 GPU (önerilen) veya A100 GPU
- **Yüksek RAM:** **AÇIK** ← bu kritik, yoksa CICIoT2023 yüklenemez

**Tahmini toplam süre:** L4 + Yüksek RAM ile **45–90 dakika** • A100 ile **25–50 dakika**

---

## 📊 Kullanılan Veri Setleri

| # | Veri Seti | Yıl | Boyut | Saldırı Sınıfı | Kaggle Slug | Online Erişim |
|---|-----------|-----|-------|----------------|-------------|----------------|
| 1 | **RT-IoT2022** | 2022 | ~70 MB | 12 saldırı tipi | `supplejade/rt-iot2022real-time-internet-of-things` | ✅ Stabil, tam yüklenir |
| 2 | **Edge-IIoTset** | 2022 | ~280 MB (ML versiyonu) | 14 saldırı + normal | `mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot` | ✅ Stabil, ML CSV tam yüklenir |
| 3 | **CICIoT2023** | 2023 | ~13 GB (169 CSV) | 7 kategori / 33 alt-sınıf | `akashdogra/cic-iot-2023` | ⚠️ Büyük; stratified subset alınır (paper-grade) |

### Online Erişim İrdelemesi
- **kagglehub** Colab'da resmi olarak desteklenir; `kaggle.json` token'ı ile her üç set de doğrudan çekilebilir.
- **İndirme süresi:** ~3 (RT-IoT2022) + ~10 dk (Edge-IIoTset) + ~25–40 dk (CICIoT2023) ≈ **40–55 dk indirme**.
- Disk 225 GB, sorun yok. RAM 51 GB (Yüksek RAM), CICIoT2023 için yine subset gerekir.
- **Olası sorun:** Slug taşınması (404). Yedek slug'lar son hücrede listelendi.

---

## 🧪 Modeller (state-of-the-art IoT IDS literatüründen)
1. **XGBoost** — GPU üzerinde `hist + device='cuda'` (projenin ana modeli)
2. **LightGBM** — Microsoft'un gradient boosting alternatifi
3. **Random Forest** — Klasik bagging baseline

**Akademik hyperparametreler:** `n_estimators=500`, `max_depth=15`, `learning_rate=0.05`

## 📐 Metrikler
- **Birincil (posterde):** Accuracy, Precision (macro), Recall (macro), F1-Score (macro), Eğitim Süresi (s)
- **Akademik ek:** Matthews Correlation Coefficient (MCC), Cohen's Kappa, ROC-AUC (macro OvR)
- **Çapraz doğrulama:** 3-fold Stratified K-Fold
- **Bonus:** Confusion Matrix + SHAP feature importance (XGBoost için)

## 1. Kütüphane Kurulumu ve Donanım Doğrulama

In [ ]:
!pip install -q kagglehub xgboost lightgbm scikit-learn shap pandas numpy matplotlib seaborn

In [ ]:
# Donanım kontrolü — GPU ve RAM doğrula
import subprocess, psutil
print('--- GPU Bilgisi ---')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout)
print('--- Sistem RAM ---')
print(f'Toplam: {psutil.virtual_memory().total / 1e9:.1f} GB')
print('  → 12 GB civarındaysa Yüksek RAM AÇIK DEĞİL — Runtime ayarlarından açın!')
print('  → 30+ GB ise Yüksek RAM açık, devam edebilirsiniz.')

In [ ]:
# Ortak importlar
import os, time, glob, warnings, json, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              matthews_corrcoef, cohen_kappa_score, roc_auc_score,
                              confusion_matrix)
from sklearn.preprocessing import label_binarize
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('✓ Kütüphaneler hazır.')

## 2. Kaggle API Token Yükleme

In [ ]:
# kaggle.json yükleme — kaggle.com → Account → Create New API Token
from google.colab import files
print('kaggle.json dosyasını yükleyin...')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

with open(os.path.expanduser('~/.kaggle/kaggle.json')) as f:
    creds = json.load(f)
os.environ['KAGGLE_USERNAME'] = creds['username']
os.environ['KAGGLE_KEY'] = creds['key']
print('✓ Kaggle API hazır.')

## 3. Veri Setlerini İndir

Tahmini süreler: RT-IoT2022 ~3 dk, Edge-IIoTset ~10 dk, **CICIoT2023 ~25–40 dk** (13 GB).

In [ ]:
import kagglehub

DATASETS = {
    'RT-IoT2022':    'supplejade/rt-iot2022real-time-internet-of-things',
    'Edge-IIoTset':  'mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot',
    'CICIoT2023':    'akashdogra/cic-iot-2023',
}

paths = {}
for name, slug in DATASETS.items():
    t0 = time.time()
    print(f'\n→ {name} ({slug}) indiriliyor ...')
    try:
        paths[name] = kagglehub.dataset_download(slug)
        print(f'  ✓ Tamam ({time.time()-t0:.0f}s) — {paths[name]}')
    except Exception as e:
        print(f'  ✗ Hata: {e}')
        paths[name] = None

print('\n--- Özet ---')
for k, v in paths.items():
    print(f'{k}: {v if v else "İNDİRİLEMEDİ"}')

In [ ]:
# Her veri setindeki CSV envanteri — boyutları göster
for name, p in paths.items():
    if p is None: continue
    csvs = sorted(glob.glob(os.path.join(p, '**', '*.csv'), recursive=True),
                  key=os.path.getsize, reverse=True)
    total_gb = sum(os.path.getsize(c) for c in csvs) / 1e9
    print(f'\n[{name}] {len(csvs)} CSV, toplam {total_gb:.2f} GB')
    for c in csvs[:5]:
        print(f'  • {os.path.basename(c):60s} ({os.path.getsize(c)/1e6:7.1f} MB)')

## 4. Veri Yükleme ve Ön İşleme

**Strateji:**
- RT-IoT2022, Edge-IIoTset → **tam veri** yüklenir.
- CICIoT2023 → 169 CSV'den **stratified subset** (her sınıftan oransal ~500K-1M satır).
- Tüm setlerde: Inf/NaN temizliği, kimlik sütunlarını drop, kategorik encode, stratified 80/20 split.

In [ ]:
LABEL_CANDIDATES = [
    'Attack_type', 'Attack_label', 'attack_cat', 'attack_type',
    'Label', 'label', 'Class', 'class', 'Target', 'target', 'category'
]
DROP_COLS = [
    'id', 'ID', 'flow_id', 'Flow ID', 'Timestamp', 'timestamp',
    'Source IP', 'Destination IP', 'src_ip', 'dst_ip',
    'Src IP', 'Dst IP', 'Source MAC', 'Destination MAC',
    'mac', 'ip.src_host', 'ip.dst_host', 'arp.src.proto_ipv4',
    'arp.dst.proto_ipv4', 'http.file_data', 'http.request.full_uri',
    'icmp.transmit_timestamp', 'http.request.uri.query', 'tcp.options',
    'tcp.payload', 'http.request.method', 'http.referer',
    'http.request.version', 'dns.qry.name.len', 'mqtt.conack.flags',
    'mqtt.protoname', 'mqtt.topic'
]

In [ ]:
def load_rt_iot2022(path):
    """RT-IoT2022 — küçük, tam yükle."""
    csvs = sorted(glob.glob(os.path.join(path, '**', '*.csv'), recursive=True),
                  key=os.path.getsize, reverse=True)
    df = pd.read_csv(csvs[0], low_memory=False)
    print(f'  RT-IoT2022 tam veri: {df.shape}')
    return df

def load_edge_iiotset(path):
    """Edge-IIoTset — ML-EdgeIIoT CSV'sini tam yükle (DNN versiyonu çok büyük)."""
    csvs = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
    ml_csv = next((c for c in csvs if 'ML-' in os.path.basename(c) or 'ML_' in os.path.basename(c)),
                  min(csvs, key=os.path.getsize))  # en küçüğü = ML versiyonu
    df = pd.read_csv(ml_csv, low_memory=False)
    print(f'  Edge-IIoTset tam veri ({os.path.basename(ml_csv)}): {df.shape}')
    return df

def load_ciciot2023(path, target_rows=1_000_000):
    """CICIoT2023 — 169 CSV'den stratified subset (her CSV'den oransal)."""
    csvs = sorted(glob.glob(os.path.join(path, '**', '*.csv'), recursive=True))
    print(f'  CICIoT2023: {len(csvs)} CSV bulundu, stratified subset alınıyor...')
    rows_per_csv = max(2000, target_rows // max(1, len(csvs)))
    dfs = []
    for i, c in enumerate(csvs):
        try:
            d = pd.read_csv(c, low_memory=False, nrows=rows_per_csv * 2)
            if len(d) > rows_per_csv:
                d = d.sample(n=rows_per_csv, random_state=RANDOM_STATE)
            dfs.append(d)
            if (i+1) % 30 == 0:
                print(f'    {i+1}/{len(csvs)} CSV okundu...')
        except Exception as e:
            print(f'    ⚠ {os.path.basename(c)}: {e}')
    df = pd.concat(dfs, ignore_index=True)
    print(f'  CICIoT2023 birleştirilmiş subset: {df.shape}')
    del dfs; gc.collect()
    return df

DATASET_LOADERS = {
    'RT-IoT2022': load_rt_iot2022,
    'Edge-IIoTset': load_edge_iiotset,
    'CICIoT2023': load_ciciot2023,
}
print('✓ Yükleyiciler hazır.')

In [ ]:
def preprocess(df, dataset_name):
    """Ortak ön işleme: etiket bul, temizle, encode et, split yap."""
    label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), df.columns[-1])
    print(f'  Etiket: "{label_col}"')

    drop_now = [c for c in DROP_COLS if c in df.columns]
    df = df.drop(columns=drop_now)
    df = df.replace([np.inf, -np.inf], np.nan).dropna()

    counts = df[label_col].value_counts()
    valid = counts[counts >= 10].index
    df = df[df[label_col].isin(valid)]
    print(f'  Temizlendi: {df.shape}, sınıf={df[label_col].nunique()}')

    y = df[label_col].astype(str)
    X = df.drop(columns=[label_col])

    for col in X.select_dtypes(include=['object']).columns:
        try:
            X[col] = LabelEncoder().fit_transform(X[col].astype(str))
        except Exception:
            X = X.drop(columns=[col])
    X = X.fillna(0)
    y_enc = LabelEncoder().fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_enc, test_size=0.2, random_state=RANDOM_STATE, stratify=y_enc
    )
    print(f'  Train: {X_train.shape}, Test: {X_test.shape}')
    return X_train, X_test, y_train, y_test, X.columns.tolist()

print('✓ Ön işleme fonksiyonu hazır.')

## 5. Modeller — Akademik Hyperparametreler

In [ ]:
def get_models(num_classes):
    """Güçlü hyperparametrelerle 3 model."""
    return {
        'XGBoost': XGBClassifier(
            n_estimators=500, max_depth=15, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            tree_method='hist', device='cuda',
            eval_metric='mlogloss', random_state=RANDOM_STATE,
            verbosity=0
        ),
        'LightGBM': LGBMClassifier(
            n_estimators=500, max_depth=15, learning_rate=0.05,
            num_leaves=127, subsample=0.9, colsample_bytree=0.9,
            num_class=num_classes, objective='multiclass',
            random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=500, max_depth=15,
            min_samples_split=5, min_samples_leaf=2,
            random_state=RANDOM_STATE, n_jobs=-1
        ),
    }

def evaluate_model(model, X_train, y_train, X_test, y_test, num_classes):
    """Eğit + tüm metrikleri hesapla."""
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    t1 = time.time()
    y_pred = model.predict(X_test)
    pred_time = time.time() - t1

    try:
        y_proba = model.predict_proba(X_test)
        if num_classes > 2:
            y_test_bin = label_binarize(y_test, classes=np.arange(num_classes))
            roc_auc = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')
        else:
            roc_auc = roc_auc_score(y_test, y_proba[:, 1])
    except Exception:
        roc_auc = np.nan

    return {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'Recall':    recall_score(y_test, y_pred, average='macro', zero_division=0),
        'F1-Score':  f1_score(y_test, y_pred, average='macro', zero_division=0),
        'MCC':       matthews_corrcoef(y_test, y_pred),
        'Kappa':     cohen_kappa_score(y_test, y_pred),
        'ROC-AUC':   roc_auc,
        'Eğitim Süresi (s)': train_time,
        'Tahmin Süresi (s)': pred_time,
        '_y_pred': y_pred,
    }

print('✓ Model + değerlendirme hazır.')

## 6. Ana Deney Döngüsü — 3 Veri Seti × 3 Model

In [ ]:
all_results = []
confusion_data = {}
best_xgb = None
best_xgb_data = None

for ds_name, ds_path in paths.items():
    if ds_path is None:
        print(f'\n⚠ {ds_name} atlandı.\n')
        continue

    print(f'\n{"="*60}\n  {ds_name} — yükleniyor\n{"="*60}')
    try:
        df = DATASET_LOADERS[ds_name](ds_path)
        X_train, X_test, y_train, y_test, feat_names = preprocess(df, ds_name)
        del df; gc.collect()
    except Exception as e:
        print(f'⚠ Ön işleme hatası: {e}')
        continue

    num_classes = len(np.unique(y_train))
    for model_name, model in get_models(num_classes).items():
        print(f'\n→ {ds_name} | {model_name} eğitiliyor...')
        try:
            res = evaluate_model(model, X_train, y_train, X_test, y_test, num_classes)
            row = {'Veri Seti': ds_name, 'Model': model_name,
                   **{k: v for k, v in res.items() if not k.startswith('_')}}
            all_results.append(row)
            confusion_data[(ds_name, model_name)] = (y_test, res['_y_pred'])
            print(f'  ✓ Acc={res["Accuracy"]:.4f}  F1={res["F1-Score"]:.4f}  '
                  f'MCC={res["MCC"]:.4f}  Süre={res["Eğitim Süresi (s)"]:.1f}s')

            if model_name == 'XGBoost' and (best_xgb is None or res['F1-Score'] > best_xgb_data['f1']):
                best_xgb = model
                best_xgb_data = {'X_test': X_test, 'feat_names': feat_names,
                                 'ds_name': ds_name, 'f1': res['F1-Score']}
        except Exception as e:
            print(f'  ✗ Hata: {e}')

    del X_train, X_test, y_train, y_test; gc.collect()

print(f'\n{"="*60}\n  Tüm deneyler tamamlandı: {len(all_results)} sonuç\n{"="*60}')

## 7. Ana Sonuç Tablosu (Poster için)

In [ ]:
results_df = pd.DataFrame(all_results)

poster_cols = ['Veri Seti', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'Eğitim Süresi (s)']
poster_df = results_df[poster_cols].copy()
for c in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    poster_df[c] = poster_df[c].apply(lambda x: f'{x*100:.2f}%')
poster_df['Eğitim Süresi (s)'] = poster_df['Eğitim Süresi (s)'].apply(lambda x: f'{x:.1f}')

print('═'*80)
print('  POSTER İÇİN ANA TABLO')
print('═'*80)
print(poster_df.to_string(index=False))
print('═'*80)

results_df.to_csv('sonuclar_tam.csv', index=False)
poster_df.to_csv('sonuclar_poster.csv', index=False)
print('\n✓ sonuclar_tam.csv ve sonuclar_poster.csv kaydedildi.')

In [ ]:
# Ana tabloyu PNG olarak çiz — postere doğrudan eklenebilir
fig, ax = plt.subplots(figsize=(14, 0.6 + 0.5 * len(poster_df)))
ax.axis('off')
tbl = ax.table(cellText=poster_df.values, colLabels=poster_df.columns,
               loc='center', cellLoc='center',
               colColours=['#1f3a68'] * len(poster_df.columns))
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.15, 1.7)
for j in range(len(poster_df.columns)):
    tbl[0, j].set_text_props(color='white', weight='bold')
colors = ['#f5f7fb', '#eaeef7', '#f5f7fb']
for i, ds in enumerate(poster_df['Veri Seti'].unique()):
    idxs = poster_df.index[poster_df['Veri Seti'] == ds].tolist()
    for ri in idxs:
        for j in range(len(poster_df.columns)):
            tbl[ri + 1, j].set_facecolor(colors[i % 3])
plt.title('Saldırı Tespit Sistemi — Model Karşılaştırması',
          fontsize=14, weight='bold', pad=12)
plt.savefig('tablo_poster.png', dpi=220, bbox_inches='tight')
plt.show()
print('✓ tablo_poster.png kaydedildi.')

## 8. Genişletilmiş Akademik Tablo (rapor için)

In [ ]:
academic_df = results_df.copy()
for c in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'MCC', 'Kappa', 'ROC-AUC']:
    academic_df[c] = academic_df[c].apply(lambda x: f'{x:.4f}' if pd.notna(x) else 'N/A')
for c in ['Eğitim Süresi (s)', 'Tahmin Süresi (s)']:
    academic_df[c] = academic_df[c].apply(lambda x: f'{x:.2f}')

print('═'*100)
print('  GENİŞLETİLMİŞ AKADEMİK TABLO (raporda/makalede kullanın)')
print('═'*100)
print(academic_df.to_string(index=False))
print('═'*100)

## 9. Çapraz Doğrulama (3-fold Stratified K-Fold)

Akademik standart — sonuçların varyansını ölçer.

In [ ]:
from sklearn.model_selection import cross_val_score

cv_results = []
print('3-fold Stratified CV — F1-Score (macro)\n')

for ds_name, ds_path in paths.items():
    if ds_path is None: continue
    print(f'--- {ds_name} ---')
    try:
        df = DATASET_LOADERS[ds_name](ds_path)
        X_train, X_test, y_train, y_test, _ = preprocess(df, ds_name)
        X_full = pd.concat([X_train, X_test])
        y_full = np.concatenate([y_train, y_test])
        del df, X_train, X_test, y_train, y_test; gc.collect()
    except Exception as e:
        print(f'  Hata: {e}'); continue

    num_classes = len(np.unique(y_full))
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    for mname, m in get_models(num_classes).items():
        t0 = time.time()
        scores = cross_val_score(m, X_full, y_full, cv=skf, scoring='f1_macro', n_jobs=1)
        dur = time.time() - t0
        cv_results.append({
            'Veri Seti': ds_name, 'Model': mname,
            'F1 Ortalama': scores.mean(), 'F1 Std': scores.std(),
            'Toplam Süre (s)': dur,
        })
        print(f'  {mname:14s}  F1 = {scores.mean():.4f} ± {scores.std():.4f}   ({dur:.1f}s)')
    del X_full, y_full; gc.collect()

cv_df = pd.DataFrame(cv_results)
cv_df.to_csv('cv_sonuclar.csv', index=False)
print('\n✓ cv_sonuclar.csv kaydedildi.')

## 10. Görselleştirmeler

In [ ]:
# F1-Score karşılaştırması
fig, ax = plt.subplots(figsize=(11, 5))
pivot = results_df.pivot(index='Veri Seti', columns='Model', values='F1-Score')
pivot.plot(kind='bar', ax=ax, color=['#2563eb', '#10b981', '#f59e0b'], edgecolor='black')
ax.set_ylabel('F1-Score (macro)')
ax.set_ylim(0, 1.0)
ax.set_title('Veri Seti × Model — F1-Score Karşılaştırması', weight='bold')
ax.legend(title='Model'); ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig('f1_karsilastirma.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Eğitim süresi karşılaştırması (kısıtlı donanım odaklı)
fig, ax = plt.subplots(figsize=(11, 5))
pivot_t = results_df.pivot(index='Veri Seti', columns='Model', values='Eğitim Süresi (s)')
pivot_t.plot(kind='bar', ax=ax, color=['#2563eb', '#10b981', '#f59e0b'], edgecolor='black')
ax.set_ylabel('Eğitim Süresi (s)')
ax.set_title('Veri Seti × Model — Eğitim Süresi', weight='bold')
ax.legend(title='Model'); ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig('sure_karsilastirma.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrix — XGBoost için her veri setinde
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, ds_name in enumerate(['RT-IoT2022', 'Edge-IIoTset', 'CICIoT2023']):
    key = (ds_name, 'XGBoost')
    if key not in confusion_data:
        axes[i].set_title(f'{ds_name} — veri yok'); axes[i].axis('off'); continue
    y_true, y_pred = confusion_data[key]
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=False, cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f'{ds_name} — XGBoost\n(normalize)', weight='bold')
    axes[i].set_xlabel('Tahmin'); axes[i].set_ylabel('Gerçek')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

## 11. SHAP Analizi — XGBoost'un "neden" verdiği kararı görelim 🎯

Bu posterinin **XGBoost-SHAP-SLM** bütününün ilk parçası: SHAP, hangi özelliklerin saldırı tespitinde belirleyici olduğunu açıklar — yani XAI.

In [ ]:
if best_xgb is not None:
    import shap
    print(f'SHAP analizi: en iyi XGBoost ({best_xgb_data["ds_name"]})')
    sample_X = best_xgb_data['X_test'].sample(n=min(2000, len(best_xgb_data['X_test'])),
                                                random_state=RANDOM_STATE)
    explainer = shap.TreeExplainer(best_xgb)
    shap_values = explainer.shap_values(sample_X)

    if isinstance(shap_values, list):
        mean_abs = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
    else:
        if shap_values.ndim == 3:
            mean_abs = np.abs(shap_values).mean(axis=(0, 2))
        else:
            mean_abs = np.abs(shap_values).mean(axis=0)

    imp_df = pd.DataFrame({'Özellik': best_xgb_data['feat_names'][:len(mean_abs)],
                            'SHAP Önem': mean_abs}).sort_values('SHAP Önem', ascending=True).tail(15)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(imp_df['Özellik'], imp_df['SHAP Önem'], color='#1f3a68')
    ax.set_xlabel('Ortalama |SHAP Değeri|')
    ax.set_title(f'XGBoost — En Etkili 15 Özellik ({best_xgb_data["ds_name"]})',
                 weight='bold')
    plt.tight_layout()
    plt.savefig('shap_onemi.png', dpi=200, bbox_inches='tight')
    plt.show()
    print('✓ shap_onemi.png kaydedildi.')
else:
    print('SHAP için XGBoost modeli bulunamadı.')

## 12. Tüm Dosyaları İndir

In [ ]:
from google.colab import files as colab_files
downloads = ['sonuclar_poster.csv', 'sonuclar_tam.csv', 'cv_sonuclar.csv',
             'tablo_poster.png', 'f1_karsilastirma.png', 'sure_karsilastirma.png',
             'confusion_matrix.png', 'shap_onemi.png']
for f in downloads:
    if os.path.exists(f):
        colab_files.download(f)
        print(f'⬇ {f}')

---

## 📌 Yedek Slug'lar (404 hatası alırsanız)

Kaggle bazen dataset taşıyor. `DATASETS` sözlüğündeki slug'ı şunlardan biriyle değiştirin:

| Set | Yedek Slug 1 | Yedek Slug 2 |
|-----|--------------|--------------|
| RT-IoT2022 | `subhajournal/iot-network-intrusion-dataset` | `harshamoturi/iot-intrusion-detection-dataset` |
| CICIoT2023 | `madhavmalhotra003/cic-iot-2023` | `dhoogla/cicioniot2023` |
| Edge-IIoTset | `nicolaspucheu/iiotset` | — |

## 🔧 Olası Sorunlar ve Çözümleri

1. **OOM (bellek dolu):** Yüksek RAM açık değil → Runtime ayarlarından açın.
2. **CICIoT2023 çok yavaş:** `load_ciciot2023(path, target_rows=500_000)` ile satır sayısını düşürün.
3. **GPU bulunamadı:** Runtime → L4 veya A100 seçin, yeniden başlatın.
4. **kaggle.json hatası:** kaggle.com → Account → "Expire API Token" → "Create New" ile yenileyin.
